# UdeA Insurance — Reto Tarifación Seguro de Salud
## Avance 1: Limpieza de Datos y Análisis Exploratorio

### Bases de datos
- **BD_Exposicion:** Periodos de cobertura de cada asegurado 
- **BD_SocioDemografica:** Perfil demográfico y condiciones preexistentes 
- **BD_Siniestros:** Reclamaciones médicas históricas 

In [1]:
# Librerías estándar de análisis de datos
import pandas as pd
import numpy as np
from pathlib import Path

# Visualización
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

print('Librerías cargadas correctamente.')

Librerías cargadas correctamente.


In [2]:
#Obtener la raíz del proyecto (subiendo un nivel desde la carpeta src/)
proyecto_root = Path.cwd().parent

# Ruta a la carpeta data
data_dir = proyecto_root / "data"

## 2. Limpieza: BD_Exposicion

### ¿Qué hace esta base de datos?
Registra **cuánto tiempo estuvo asegurada cada persona**. Esto es fundamental
porque en actuaría no importa solo cuánto costó una persona, sino cuánto costó
*por unidad de tiempo expuesta*. Ese indicador se llama **tasa de siniestralidad**.

### Pasos que vamos a ejecutar:
1. Diagnóstico inicial (tipos, nulos, duplicados)
2. Conversión de fechas (de texto a datetime)
3. Calcular la **fecha de salida real** (cancelación o fin de contrato)
4. Calcular la **exposición** en días y meses
5. Detectar y reportar inconsistencias

In [3]:
# Leer uno de los archivos 
df_exposicion = pd.read_csv(data_dir / "BD_Exposicion.txt", sep="\t")  # o sep=',' o sep=';'

# ── 2.1 Diagnóstico inicial ──────────────────────────────────────────────────
print('Primeras cinco filas ')
display(df_exposicion.head(5))

print('\n Tipos de datos y nulos por columna')
print(df_exposicion.info())

print('\n Valores nulos por columna y porcentaje')
nulos = df_exposicion.isnull().sum()
pct   = (nulos / len(df_exposicion) * 100).round(2)
print(pd.DataFrame({'nulos': nulos, 'pct_%': pct}))

print(f'\nFilas duplicadas: {df_exposicion.duplicated().sum():,}')
print(f'Asegurados únicos (Asegurado_Id): {df_exposicion["Asegurado_Id"].nunique():,}')
print(f'Pólizas únicas  (Poliza_Asegurado_Id): {df_exposicion["Poliza_Asegurado_Id"].nunique():,}')


Primeras cinco filas 


,Asegurado_Id,Poliza_Asegurado_Id,FECHA_INICIO,FECHA_CANCELACION,FECHA_FIN
0,61630146,271191574,29/01/2024,29/06/2024,29/06/2024
1,7966225,283714899,13/11/2024,NaN,31/12/2024
2,3124492,50801309,1/01/2024,17/01/2024,17/01/2024
3,11863430,236507980,1/01/2024,NaN,31/12/2024
4,4345756,230314288,1/01/2024,NaN,31/12/2024



 Tipos de datos y nulos por columna
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 320002 entries, 0 to 320001
Data columns (total 5 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   Asegurado_Id         320002 non-null  int64 
 1   Poliza_Asegurado_Id  320002 non-null  int64 
 2   FECHA_INICIO         320002 non-null  object
 3   FECHA_CANCELACION    109031 non-null  object
 4   FECHA_FIN            320002 non-null  object
dtypes: int64(2), object(3)
memory usage: 12.2+ MB
None

 Valores nulos por columna y porcentaje
                      nulos  pct_%
Asegurado_Id              0   0.00
Poliza_Asegurado_Id       0   0.00
FECHA_INICIO              0   0.00
FECHA_CANCELACION    210971  65.93
FECHA_FIN                 0   0.00

Filas duplicadas: 0
Asegurados únicos (Asegurado_Id): 296,020
Pólizas únicas  (Poliza_Asegurado_Id): 320,002


In [4]:
# Convertir columnas de fecha antes de operar con ellas
for columna in ['FECHA_INICIO', 'FECHA_FIN', 'FECHA_CANCELACION']:
    df_exposicion[columna] = pd.to_datetime(df_exposicion[columna], errors='coerce')

print('Columnas de fecha convertidas a datetime correctamente.')

Columnas de fecha convertidas a datetime correctamente.


/var/folders/wz/5knc4f1j3y1g1v82hm80x7g00000gn/T/ipykernel_10831/1802479245.py:3: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_exposicion[columna] = pd.to_datetime(df_exposicion[columna], errors='coerce')
/var/folders/wz/5knc4f1j3y1g1v82hm80x7g00000gn/T/ipykernel_10831/1802479245.py:3: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_exposicion[columna] = pd.to_datetime(df_exposicion[columna], errors='coerce')
/var/folders/wz/5knc4f1j3y1g1v82hm80x7g00000gn/T/ipykernel_10831/1802479245.py:3: UserWarning: Parsing dates in %d/%m/%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  df_exposicion[columna] = pd.to_datetime(df_exposicion[columna], errors='coerce')


## REGLA IMPORTANTE PARA CONFIGURAR LA FECHA DE FINALIZACIÓN DE CONTRARO 
Si la persona canceló antes de que terminara el contrato, su fecha de salida real es FECHA_CANCELACION (la más temprana). Si NO canceló, su fecha de salida es FECHA_FIN.

- Crear nueva columna llamada FECHA_SALIDA

In [5]:
# ── 2.3 Fecha de salida real ─────────────────────────────────────────────────

df_exposicion['FECHA_SALIDA'] = df_exposicion['FECHA_CANCELACION'].fillna(df_exposicion['FECHA_FIN'])

# Validación extra: si FECHA_CANCELACION ES MAYOR A  FECHA_FIN, hay un error en los datos
# (la cancelación ocurrió DESPUÉS del fin del contrato, lo cual no tiene sentido)
mask_error = (
    df_exposicion['FECHA_CANCELACION'].notna() &
    (df_exposicion['FECHA_CANCELACION'] > df_exposicion['FECHA_FIN'])
)
print(f'Registros con FECHA_CANCELACION > FECHA_FIN (inconsistentes): {mask_error.sum():,}')

# Para esos casos, corregimos usando FECHA_FIN como salida real
df_exposicion.loc[mask_error, 'FECHA_SALIDA'] = df_exposicion.loc[mask_error, 'FECHA_FIN']

print('\n Ejemplo de la nueva columna FECHA_SALIDA:')
display(df_exposicion[['Asegurado_Id', 'FECHA_INICIO', 'FECHA_CANCELACION', 'FECHA_FIN', 'FECHA_SALIDA']].head(8))

Registros con FECHA_CANCELACION > FECHA_FIN (inconsistentes): 18

 Ejemplo de la nueva columna FECHA_SALIDA:


,Asegurado_Id,FECHA_INICIO,FECHA_CANCELACION,FECHA_FIN,FECHA_SALIDA
0,61630146,2024-01-29,2024-06-29,2024-06-29,2024-06-29
1,7966225,2024-11-13,NaT,2024-12-31,2024-12-31
2,3124492,2024-01-01,2024-01-17,2024-01-17,2024-01-17
3,11863430,2024-01-01,NaT,2024-12-31,2024-12-31
4,4345756,2024-01-01,NaT,2024-12-31,2024-12-31
5,8134114,2024-01-01,NaT,2024-12-31,2024-12-31
6,34285636,2024-01-01,2024-02-18,2024-02-18,2024-02-18
7,54470034,2024-01-01,2024-12-31,2024-12-31,2024-12-31


# CALCULO DE EXPOSIÓN

La exposición es la cantidad de tiempo que la persona estuvo 'en riesgo'.
La calculamos en días y en meses (meses es más útil para calcular primas mensuales).

In [6]:
# ── 2.4 Calcular exposición ──────────────────────────────────────────────────

df_exposicion['dias_expuesto'] = (df_exposicion['FECHA_SALIDA'] - df_exposicion['FECHA_INICIO']).dt.days

# Convertir a meses (aproximación: 1 mes = 30.4375 días)
df_exposicion['meses_expuesto'] = df_exposicion['dias_expuesto'] / 30.4375

# Detectar exposiciones inválidas
print(' Diagnóstico de exposición ')
print(f'Exposición negativa (error de fechas):  {(df_exposicion["dias_expuesto"] < 0).sum():,}')
print(f'Exposición = 0 días:                    {(df_exposicion["dias_expuesto"] == 0).sum():,}')
print(f'Exposición > 365 días (más de 1 año):   {(df_exposicion["dias_expuesto"] > 365).sum():,}')
print()
print(df_exposicion['dias_expuesto'].describe())

 Diagnóstico de exposición 
Exposición negativa (error de fechas):  0
Exposición = 0 días:                    6,578
Exposición > 365 días (más de 1 año):   0

count    320002.000000
mean        294.559943
std         116.671127
min           0.000000
25%         244.000000
50%         365.000000
75%         365.000000
max         365.000000
Name: dias_expuesto, dtype: float64


In [15]:
# ── 2.5 Cierre final de BD_Exposicion (modelado) ────────────────────────────
# 1) Filtrar exposición válida
n_invalidos = (df_exposicion['dias_expuesto'] <= 0).sum()
pct_invalidos = n_invalidos / len(df_exposicion) * 100

print(f'Exposición <= 0: {n_invalidos:,} ({pct_invalidos:.2f}%)')

df_exposicion['exposicion_valida'] = df_exposicion['dias_expuesto'] > 0
df_exposicion_limpio = df_exposicion[df_exposicion['exposicion_valida']].copy()

# 2) Diagnóstico de duplicados
dup_exactos = df_exposicion_limpio.duplicated().sum()
dup_poliza_fechas = df_exposicion_limpio.duplicated(
    subset=['Poliza_Asegurado_Id', 'FECHA_INICIO', 'FECHA_SALIDA']
).sum()
dup_poliza = df_exposicion_limpio.duplicated(subset=['Poliza_Asegurado_Id']).sum()

print('\n=== Resumen final de BD_Exposicion ===')
print(f'Duplicados exactos: {dup_exactos:,}')
print(f'Duplicados por póliza+fechas: {dup_poliza_fechas:,}')
print(f'Pólizas repetidas (cualquier periodo): {dup_poliza:,}')

# 3) Base final de modelado (remueve solo duplicado exacto)
df_exposicion_modelo = df_exposicion_limpio.drop_duplicates().copy()

print('\n=== información final para modelado ===')
print(f'Filas finales: {len(df_exposicion_modelo):,}')
print(f'Asegurados únicos: {df_exposicion_modelo["Asegurado_Id"].nunique():,}')
print(f'Pólizas únicas: {df_exposicion_modelo["Poliza_Asegurado_Id"].nunique():,}')

Exposición <= 0: 6,578 (2.06%)

=== Resumen final de BD_Exposicion ===
Duplicados exactos: 0
Duplicados por póliza+fechas: 0
Pólizas repetidas (cualquier periodo): 0

=== información final para modelado ===
Filas finales: 313,424
Asegurados únicos: 293,288
Pólizas únicas: 313,424


## 3. Limpieza: BD_SocioDemografica

### ¿Qué hace esta base de datos?
Contiene el **perfil de riesgo** de cada asegurado. Aquí están las variables que
más influyen en el costo de un seguro de salud: edad, sexo, ciudad y enfermedades
preexistentes. Estas serán las principales **variables predictoras** en el modelo.

### Pasos a ejeciutar:
1. Diagnóstico inicial
2. Convertir y calcular edad desde FechaNacimiento 
3. Crear grupos de edad (variable clave para tarifación)
4. Validar variables binarias de condiciones
5. Normalizar la variable CIUDAD

In [5]:
# Leer la base de siniestros (ajusta el separador si hace falta)
df_siniestros = pd.read_csv(data_dir / "BD_Siniestros.txt", sep="\t", encoding="latin1")

# Ver las primeras filas
df_siniestros.head()

,Fecha_Reclamacion,Afiliado_Id,Reclamacion,Diagnostico_Codigo,Diagnostico_Desc,Eventos,Valor_Pagado
0,4/12/2024,31586355,CONSULTA MEDICA,9,DIAGNÓSTICO PENDIENTE,1,101576.02545
1,15/01/2024,22753623,CONSULTA MEDICA,9,DIAGNÓSTICO PENDIENTE,1,80630.89152
2,27/05/2024,29321930,CONSULTA MEDICA,9,DIAGNÓSTICO PENDIENTE,1,90237.30633
3,6/11/2024,5258725,CONSULTA MEDICA,9,DIAGNÓSTICO PENDIENTE,1,90237.30633
4,19/07/2024,22815238,CONSULTA MEDICA,9,DIAGNÓSTICO PENDIENTE,1,8346.55713


In [14]:
df_siniestros.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1919949 entries, 0 to 1919948
Data columns (total 7 columns):
 #   Column              Dtype  
---  ------              -----  
 0   Fecha_Reclamacion   object 
 1   Afiliado_Id         int64  
 2   Reclamacion         object 
 3   Diagnostico_Codigo  object 
 4   Diagnostico_Desc    object 
 5   Eventos             int64  
 6   Valor_Pagado        float64
dtypes: float64(1), int64(2), object(4)
memory usage: 102.5+ MB


In [6]:
# Leer la base sociodemográfica (ajusta el separador si hace falta)
df_sociodemografica = pd.read_csv(data_dir / "BD_SocioDemografica.txt", sep="\t", encoding="latin1")

# Ver las primeras filas
df_sociodemografica.head()

,Afiliado_Id,Sexo_Cd,FechaNacimiento,CIUDAD,CANCER,DIABETES,ENF_CARDIACA,HIPERTENSION,ENF_PULMONAR
0,921437,F,30/04/68,Medellin,0,0,0,0,0
1,60504878,M,18/02/12,Medellin,0,0,0,0,0
2,55074222,F,23/10/14,Medellin,0,0,0,0,0
3,23690690,F,27/06/89,Cartagena,0,0,0,0,0
4,45506882,M,30/06/09,Cali,0,0,0,0,0


In [12]:
df_sociodemografica.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 261267 entries, 0 to 261266
Data columns (total 9 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   Afiliado_Id      261267 non-null  int64 
 1   Sexo_Cd          261267 non-null  object
 2   FechaNacimiento  261267 non-null  object
 3   CIUDAD           261267 non-null  object
 4   CANCER           261267 non-null  int64 
 5   DIABETES         261267 non-null  int64 
 6   ENF_CARDIACA     261267 non-null  int64 
 7   HIPERTENSION     261267 non-null  int64 
 8   ENF_PULMONAR     261267 non-null  int64 
dtypes: int64(6), object(3)
memory usage: 17.9+ MB
